# 1. import and configs

In [ ]:
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tabulate import tabulate

In [ ]:
# Fill this with your own runs.
#
# Each entry describes one `polypsense.reid.cli --task=fewshotretrieval` run:
#
#   "model"        display name used in tables and plots
#   "path"         path to the `outputs.pth` written by the evaluator, i.e.
#                  output/fewshotretrieval/<exp_name>/outputs.pth
#   "gallery_mode" optional, "global" or "same_video" (see README)
#   "checkpoint"   optional, free-form tag of the encoder checkpoint
#
# By convention below, an experiment evaluated with --fsret_gallery_mode=same_video
# is named "<exp_name>.same_video", so that both gallery modes of the same encoder
# can live side by side. Adapt to your own naming if you prefer.
#
# The model list is the one reported in the paper; drop the rows you did not run.

EXP_NAMES = {
    'Ours': "<exp_name>",
    'NTSSL': "<exp_name>",
    'TA': "<exp_name>",
    'SUP': "<exp_name>",
    'TPC (Weakly-sup)': "<exp_name>",
    'MVE (Weakly-sup)': "<exp_name>",
    'TPC (SSL)': "<exp_name>",
    'MVE (SSL)': "<exp_name>",
    'EndoFM': "<exp_name>",
    'EndoFM-LV': "<exp_name>",
    'EndoViT': "<exp_name>",
    'SurgeNet': "<exp_name>",
    'DinoV3-S': "<exp_name>",
    'DinoV3-B': "<exp_name>",
    'DinoV3-L': "<exp_name>",
    'DinoV2-S': "<exp_name>",
    'DinoV2-B': "<exp_name>",
    'DinoV2-L': "<exp_name>",
    'DinoV2-g': "<exp_name>",
    'VJEPA-2': "<exp_name>",
}


def get_runs(gallery_mode):
    suffix = "" if gallery_mode == "global" else f".{gallery_mode}"
    return [
        {
            "model": model,
            "checkpoint": exp_name,
            "path": f"output/fewshotretrieval/{exp_name}{suffix}/outputs.pth",
            "gallery_mode": gallery_mode,
        }
        for model, exp_name in EXP_NAMES.items()
    ]


In [ ]:
SHOW_GALLERY="global"
RUNS = get_runs(SHOW_GALLERY)
OUTPUT_DIR = Path("output/analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. loading utilities

In [ ]:
def torch_load(path: str | Path) -> Dict[str, Any]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    # weights_only=False is useful because the evaluator output may contain
    # dictionaries/lists/metadata, not just tensors.
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def get_run_tag(run: Dict[str, Any], key: str, default: Any = None) -> Any:
    return run[key] if key in run else default

# 3. Flatten few-shot results

In [ ]:
# The protocol reported in the paper: random support selection, scored by the
# maximum cosine similarity to any support tracklet. The evaluator writes
# results under keys of the form "k<K>/random/max_sim".
SELECTOR = "random"
SCORER = "max_sim"


def parse_fewshot_key(key: str) -> Dict[str, Any]:
    parts = key.split("/")

    if len(parts) != 3:
        raise ValueError(f"Unexpected few-shot key format: {key}")

    k_part, selector, scorer = parts

    match = re.match(r"k(\d+)", k_part)
    if match is None:
        raise ValueError(f"Could not parse k from key: {key}")

    return {
        "k": int(match.group(1)),
        "selector": selector,
        "scorer": scorer,
    }


def flatten_fewshot_results(
    data: Dict[str, Any],
    run: Dict[str, Any],
) -> pd.DataFrame:
    results = data.get("results", {})
    fewshot = results.get("fewshot", None)

    if fewshot is None:
        return pd.DataFrame()

    rows = []

    for key, metrics in fewshot.items():
        parsed = parse_fewshot_key(key)

        # Ignore any other selector/scorer, e.g. from outputs of older runs.
        if parsed["selector"] != SELECTOR or parsed["scorer"] != SCORER:
            continue

        row = {
            "model": run["model"],
            "path": str(run["path"]),
            "checkpoint": get_run_tag(run, "checkpoint", None),
            "gallery_mode": get_run_tag(run, "gallery_mode", run["gallery_mode"]),
            "split": get_run_tag(run, "split", None),
            "fewshot_key": key,
            "k": parsed["k"],
        }

        for metric_name, value in metrics.items():
            row[metric_name] = float(value)

        rows.append(row)

    return pd.DataFrame(rows)

# 4. Load all runs

In [ ]:
fewshot_dfs = []

loaded = {}

for run in RUNS:
    path = Path(run["path"])
    print(f"Loading: {run['model']} -> {path}")

    try:
        data = torch_load(path)
        loaded[str(path)] = data
    except FileNotFoundError:
        print(f"Could not find {str(path)}")
        continue  # ignore missing

    fewshot_df = flatten_fewshot_results(data, run)
    if len(fewshot_df) > 0:
        fewshot_dfs.append(fewshot_df)

fewshot_results = (
    pd.concat(fewshot_dfs, ignore_index=True)
    if len(fewshot_dfs) > 0
    else pd.DataFrame()
)

print("Few-shot rows:", len(fewshot_results))

# 5. Save raw flattened results

In [ ]:
fewshot_results.to_csv(OUTPUT_DIR / "fewshot_results_flat.csv", index=False)

# 6. Basic sanity checks

In [ ]:
def show_unique_values(df: pd.DataFrame, columns: List[str]):
    for col in columns:
        if col in df.columns:
            print(f"\n{col}:")
            print(sorted(df[col].dropna().unique()))


show_unique_values(
    fewshot_results,
    ["model", "checkpoint", "gallery_mode", "k"],
)

In [ ]:
# Check whether each configuration has valid episodes.
episode_check = (
    fewshot_results
    .groupby(["model", "gallery_mode", "k"], dropna=False)
    .agg(
        num_rows=("map", "count"),
        num_episodes=("num_episodes", "mean"),
    )
    .reset_index()
)

episode_check.sort_values("num_episodes").head(20)

# 7. Main summary table

In [ ]:
METRICS = [
    "map",
    "hr1",
    "hr5",
    "recall1",
    "recall5",
    "r_precision",
    "last_pos_rank",
    "fp_before_full_recall",
    "num_episodes",
]

available_metrics = [m for m in METRICS if m in fewshot_results.columns]

fewshot_summary = (
    fewshot_results
    .groupby(
        ["model", "checkpoint", "gallery_mode", "k"],
        dropna=False,
    )[available_metrics]
    .mean()
    .reset_index()
)

fewshot_summary.head()

# 8. Compare models

In [ ]:
def compare_models(
    summary: pd.DataFrame,
    gallery_mode: str = "same_video",
    metric_cols: Optional[List[str]] = None,
) -> pd.DataFrame:
    if metric_cols is None:
        metric_cols = ["map", "hr1", "hr5", "recall5", "r_precision", "fp_before_full_recall"]

    cols = ["model", "checkpoint", "gallery_mode", "k"] + metric_cols

    out = summary.query("gallery_mode == @gallery_mode")

    return out[cols].sort_values(["k", "map"], ascending=[True, False])


print(
    tabulate(
        compare_models(
            fewshot_summary,
            gallery_mode=SHOW_GALLERY,
        )[["model", "k", "map", "hr1"]],
        headers="keys",
        tablefmt="psql", # "latex"
        showindex=False,
    )
)

# 9. Compare same-video vs global

In [ ]:
def compare_gallery_modes(summary: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "model",
        "checkpoint",
        "gallery_mode",
        "k",
        "map",
        "hr1",
        "hr5",
        "recall5",
        "r_precision",
        "fp_before_full_recall",
        "num_episodes",
    ]

    cols = [c for c in cols if c in summary.columns]

    return summary[cols].sort_values(["model", "k", "gallery_mode"])


print(
    compare_gallery_modes(fewshot_summary)[["model", "gallery_mode", "k", "map"]]
)

# 10. Pivot tables for paper-style inspection

### mAP table

In [ ]:
def pivot_metric(
    summary: pd.DataFrame,
    metric: str = "map",
    gallery_mode: str = "same_video",
    sort_by: int = 1,  # name of the column (K: int, in this case)
) -> pd.DataFrame:
    out = summary.query("gallery_mode == @gallery_mode")

    table = out.pivot_table(
        index="model",
        columns="k",
        values=metric,
        aggfunc="mean",
    )

    table = table.sort_values(by=sort_by, ascending=False)
    table = table.mul(100)

    return table


print(
    tabulate(
        pivot_metric(
            fewshot_summary,
            metric="map",
            gallery_mode=SHOW_GALLERY,
            sort_by=1,
        ),
        headers="keys",
        tablefmt="psql", #latex 
        floatfmt=".2f",
        showindex=True,
    )
)

# 11. Plot k-shot curves

In [ ]:
def plot_kshot_curve(
    summary: pd.DataFrame,
    metric: str = "map",
    gallery_mode: str = "same_video",
    models: Optional[List[str]] = None,
):
    df = summary.query("gallery_mode == @gallery_mode")

    if models is not None:
        df = df[df["model"].isin(models)]

    fig, ax = plt.subplots(figsize=(7, 4))

    for model, group in df.groupby("model"):
        group = group.sort_values("k")
        ax.plot(group["k"], group[metric], marker="o", label=model)

    ax.set_xlabel("K shots")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} | {gallery_mode}")
    ax.set_xticks(sorted(df["k"].unique()))
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()

    return fig, ax


plot_kshot_curve(
    fewshot_summary,
    metric="map",
    gallery_mode=SHOW_GALLERY,
)